# Chapter 20: Map Representations

<a href="../lite/lab/index.html?path=ch20_map_representations.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

What is a map? To Google Maps, it is road segments and GPS coordinates. To a vacuum
robot, it is a grid of "clean" and "not yet clean" cells. To a self driving car, it is
a centimeter precise 3D model of every lane marking and curb. The representation you
choose for your map determines what your robot can and cannot do.

This chapter explores the four main map representations used in robotics and builds
a working occupancy grid mapper from scratch.

```{admonition} What you will build
:class: tip

- Build a working occupancy grid mapper that accumulates LiDAR scans into a map
- Compare landmark maps, occupancy grids, and topological maps for storage and functionality
- Understand the log-odds update rule that makes occupancy grids numerically stable
- See how hybrid representations combine the best of all map types

**Real world application:** Robot vacuums use occupancy grids. Self driving cars use 3D point clouds. Delivery robots use topological maps. After this chapter, you will know which map representation to choose for your application.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **nav2_map_server (ROS 2)** | Serves occupancy grid maps for robot navigation |
| **OctoMap** | 3D occupancy mapping using octrees (memory efficient) |
| **Open3D** | 3D point cloud and voxel grid processing |
| **slam_toolbox (ROS 2)** | Online and offline 2D SLAM with occupancy grid output |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

## 20.1 Landmarks

A **landmark map** stores a list of distinct features: their positions and possibly
descriptors (color, shape, size). This is the representation used by EKF-SLAM
and most feature based visual SLAM systems.

$$\mathcal{M} = \{(\mathbf{l}_1, d_1), (\mathbf{l}_2, d_2), \ldots, (\mathbf{l}_N, d_N)\}$$

**Pros:** Compact, efficient for data association, natural for EKF-SLAM.
**Cons:** Cannot represent free space, cannot do path planning directly.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(42)
n_landmarks = 20
room_size = 20.0
# ──────────────────────────────────────────────────────────────────────────────

landmarks = np.random.uniform(1, room_size - 1, (n_landmarks, 2))
types = np.random.choice(['tree', 'pole', 'sign', 'corner'], n_landmarks)
colors = {'tree': 'forestgreen', 'pole': 'steelblue', 'sign': 'tomato', 'corner': 'orange'}
markers = {'tree': '^', 'pole': 's', 'sign': 'D', 'corner': 'o'}

fig, ax = plt.subplots(figsize=(8, 7))
for tp in ['tree', 'pole', 'sign', 'corner']:
    mask = types == tp
    ax.scatter(landmarks[mask, 0], landmarks[mask, 1], c=colors[tp], s=100,
              marker=markers[tp], label=tp, zorder=5)
ax.set_xlim(0, room_size); ax.set_ylim(0, room_size)
ax.set_aspect('equal'); ax.legend(fontsize=10)
ax.set_title(f"Landmark map: {n_landmarks} features", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Map storage: {n_landmarks} landmarks x 2 coordinates = {n_landmarks * 2} floats")

## 20.2 Occupancy Grids

An **occupancy grid** divides the world into cells. Each cell stores the probability
that it is occupied. This is the standard representation for 2D navigation.

We use **log-odds** for numerical stability:

$$l_{i,t} = l_{i,t-1} + \log\frac{p(m_i \mid z_t)}{1 - p(m_i \mid z_t)} - l_0$$

**Inverse sensor model:** for each LiDAR ray, cells along the ray are marked **free**
(negative update), and the cell at the endpoint is marked **occupied** (positive update).

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
grid_size = 100         # cells per side
cell_size = 0.2         # meters per cell (total: 20m x 20m)
n_beams = 72            # LiDAR beams (every 5 degrees)
max_range = 8.0         # sensor max range (meters)
l_occ = 0.85            # log-odds occupied
l_free = -0.4           # log-odds free
# ──────────────────────────────────────────────────────────────────────────────

log_odds = np.zeros((grid_size, grid_size))

# Define walls
walls = np.zeros((grid_size, grid_size), dtype=bool)
walls[0, :] = walls[-1, :] = walls[:, 0] = walls[:, -1] = True
walls[20:60, 50] = True   # internal wall
walls[40, 20:50] = True   # vertical wall
walls[70:78, 25:33] = True # obstacle

def update_grid(log_odds, rx, ry, walls, n_beams, max_r_cells, l_occ, l_free):
    angles = np.linspace(0, 2*np.pi, n_beams, endpoint=False)
    for angle in angles:
        for r in range(1, int(max_r_cells)):
            cx = int(rx + r * np.cos(angle))
            cy = int(ry + r * np.sin(angle))
            if cx < 0 or cx >= grid_size or cy < 0 or cy >= grid_size:
                break
            if walls[cy, cx]:
                log_odds[cy, cx] += l_occ
                break
            else:
                log_odds[cy, cx] += l_free
    log_odds[:] = np.clip(log_odds, -5, 5)

# Scan from multiple positions
positions = [(30,30),(50,30),(70,30),(70,50),(70,70),(50,70),(30,70),(50,50)]
max_r_cells = max_range / cell_size

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for idx, pos in enumerate(positions):
    update_grid(log_odds, pos[0], pos[1], walls, n_beams, max_r_cells, l_occ, l_free)
    ax = axes[idx // 4, idx % 4]
    prob = 1 - 1 / (1 + np.exp(log_odds))
    ax.imshow(prob, cmap='gray_r', origin='lower', vmin=0, vmax=1,
              extent=[0, grid_size*cell_size, 0, grid_size*cell_size])
    for p in positions[:idx+1]:
        ax.plot(p[0]*cell_size, p[1]*cell_size, 'ro', ms=3)
    ax.set_title(f"After {idx+1} scans", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle("Occupancy grid builds up with each LiDAR scan", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Key observations:**
- Each scan adds information. Unknown areas (gray) become known (black or white).
- **Log-odds** allow clean accumulation: just add the update from each scan.
- More scans produce a more confident map.
- Occupancy grids enable **path planning** directly (A*, Dijkstra on free cells).

## 20.3 Sparse vs Dense

| Property | Sparse (landmarks) | Dense (occupancy grid) |
|----------|:---:|:---:|
| **Storage** | $O(N)$ for $N$ landmarks | $O(W \times H)$ cells |
| **Path planning** | Needs separate representation | Direct (A*, wavefront) |
| **Data association** | Feature matching | Scan matching (ICP) |
| **Typical use** | EKF-SLAM, visual SLAM | Navigation, obstacle avoidance |

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
room_meters = 50.0
n_landmarks_s = 200
# ──────────────────────────────────────────────────────────────────────────────

storage = {
    'Sparse\n(200 landmarks)': 200 * 2,
    'Grid 2D\n(5cm res)': int(room_meters/0.05)**2,
    'Grid 2D\n(2cm res)': int(room_meters/0.02)**2,
    'Voxel 3D\n(5cm, 3m high)': int(room_meters/0.05)**2 * int(3/0.05)
}

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(storage.keys(), storage.values(),
              color=['steelblue', 'tomato', 'tomato', 'orange'], edgecolor='k', alpha=0.8)
for bar, s in zip(bars, storage.values()):
    label = f'{s:,}' if s < 1e6 else f'{s/1e6:.1f}M'
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), label,
            ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylabel("Storage (floats)"); ax.set_yscale('log')
ax.set_title(f"Storage comparison for {room_meters:.0f}m x {room_meters:.0f}m environment", fontsize=13)
plt.tight_layout()
plt.show()

## 20.4 Metric vs Topological

A **metric map** stores exact positions and distances. A **topological map** stores
only the connectivity: which places are connected to which, and roughly how far apart.

Modern systems often use **hybrid** representations: a topological backbone with
local metric maps at each node.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
places = {'Kitchen': (2,1), 'Living': (5,1), 'Bedroom': (8,1),
          'Hall': (5,3.5), 'Bath': (2,3.5), 'Garage': (8,3.5)}
edges = [('Kitchen','Living'),('Living','Bedroom'),('Living','Hall'),
         ('Hall','Bath'),('Hall','Bedroom'),('Kitchen','Bath'),('Bedroom','Garage')]
# ──────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for name, pos in places.items():
    ax.plot(*pos, 'o', color='steelblue', ms=18, zorder=5)
    ax.annotate(name, xy=pos, xytext=(0, -20), textcoords='offset points', ha='center', fontsize=9, fontweight='bold')
for a, b in edges:
    pa, pb = places[a], places[b]
    d = np.sqrt((pa[0]-pb[0])**2 + (pa[1]-pb[1])**2)
    ax.plot([pa[0],pb[0]], [pa[1],pb[1]], 'gray', lw=2)
    ax.text((pa[0]+pb[0])/2, (pa[1]+pb[1])/2+0.25, f'{d:.1f}m', fontsize=8, ha='center', color='gray')
ax.set_title("Metric map (exact positions)", fontsize=13)
ax.axis('off')

# Topological: different layout, same connectivity
topo = {'Kitchen':(1,1), 'Bath':(1,3), 'Living':(3,1.5), 'Hall':(3,3.5), 'Bedroom':(5,1), 'Garage':(5,3)}
ax = axes[1]
for name, pos in topo.items():
    ax.plot(*pos, 'o', color='tomato', ms=18, zorder=5)
    ax.annotate(name, xy=pos, xytext=(0, -20), textcoords='offset points', ha='center', fontsize=9, fontweight='bold')
for a, b in edges:
    pa, pb = topo[a], topo[b]
    ax.plot([pa[0],pb[0]], [pa[1],pb[1]], 'gray', lw=2)
ax.set_title("Topological map (connectivity only)", fontsize=13)
ax.axis('off')

plt.tight_layout()
plt.show()

**Key observations:**
- **Landmark maps** are compact but cannot represent free space.
- **Occupancy grids** represent free space explicitly, enabling path planning.
- **Topological maps** are the most compact and support high level reasoning.
- **Hybrid** representations combine the best of all worlds.

---

## Exercises

### Exercise 20.1: Build an occupancy grid from scratch

Implement a 100x100 occupancy grid for a 10m x 10m room. Place walls on all sides
and add two internal walls. Simulate LiDAR scans from 5 positions. Display the final map.

In [ ]:
# Your code here
# 1. Initialize log_odds = np.zeros((100, 100))
# 2. Define walls as a boolean grid
# 3. For each robot position, update the grid using the inverse sensor model
# 4. Convert to probability and display

### Exercise 20.2: Path planning on occupancy grid

Using the completed grid from Exercise 20.1:
1. Threshold: cells with p > 0.6 are occupied, p < 0.4 are free
2. Implement BFS from start (10,10) to goal (80,80)
3. Plot the path on the map

In [ ]:
# Your code here

### Exercise 20.3: Compare map representations (challenge)

For a 100m x 100m environment, compute and plot the storage needed for:
(a) Occupancy grid at 1cm, 5cm, 10cm, 50cm resolution
(b) 3D voxel grid at 5cm (10m height)
(c) 100, 500, 1000, 5000, 10000 point landmarks
At what landmark count does sparse become larger than a 10cm grid?

In [ ]:
# Your code here